# Remember: post-training in het klein

**Doel:** doorloop zelf data → nulmeting → training → evaluatie → opslaan → opnieuw gebruiken.

Het model vergelijkt een korte gespreksuitspraak met eerder vastgelegde context.
Het antwoordt `MELD` bij een nog niet opgeloste tegenstrijdigheid over een deadline,
en anders `STIL`. Dat is één bouwsteen van de toekomstige gespreksassistent.

Alle projecten en gesprekken zijn fictief. Deze kleine tekstproef leert je de
trainingsketen; hij bewijst nog geen verbetering in echte gesprekken of goede timing.
De uitkomst mag beter, gelijk of slechter zijn dan de nulmeting.

## Waar draait wat?

| Onderdeel | Plaats | Functie |
|---|---|---|
| Dit bestand | Remember-repository; upload een kopie naar Colab | Code en uitleg bewaren |
| Google Colab | Python-runtime op een GPU bij Google | Code uitvoeren, trainen en testen |
| Hugging Face Hub | Modelcatalogus | De bestaande modelgewichten downloaden |
| Transformers + PyTorch | In de Colab-runtime | Model laden en berekeningen uitvoeren |
| TRL + PEFT | In de Colab-runtime | SFT-training uitvoeren met LoRA-adapters |
| Python-lijsten en JSON-bestanden | In de Colab-runtime | Data, benchmark en resultaten |

We gebruiken `Qwen/Qwen2.5-0.5B-Instruct`: een bestaand, klein instructiemodel van
ongeveer een half miljard parameters. Dit is een leerkeuze, geen keuze voor productie.

**Begin:** open https://colab.research.google.com, kies *File → Upload notebook*,
upload dit bestand en kies *Runtime → Change runtime type → GPU* (bijvoorbeeld T4).
Gebruik een verse runtime en voer de cellen op volgorde uit met de afspeelknop of
Shift+Enter. Een Google-account is nodig. De publieke modeldownload vereist geen
Hugging Face-token. Colab heeft gratis GPU-toegang, maar beschikbaarheid is niet gegarandeerd.

Je Mac is tijdens deze oefening de editor/browser; de Python-code draait op de Colab-machine.

## 1. Gereedschap installeren

Deze cel installeert Python-libraries in Colab. PyTorch wordt door de GPU-runtime
geleverd. We zetten de belangrijkste libraryversies vast; de daadwerkelijk gebruikte
versies en modelrevisie worden bij de resultaten opgeslagen.

Als Colab na installatie om een runtime-herstart vraagt: herstart en ga verder met
cel 2. Installeer voordat je de libraries in andere cellen importeert.

In [ ]:
%pip install -q "transformers==4.56.2" "trl==0.23.1" "peft==0.17.1" "datasets==4.0.0" "accelerate==1.10.1"

## 2. De run vastleggen

Een **run** is één experiment met vaste data, instellingen en modelgewichten.
De cel toont de beschikbare GPU. Iedere volledige doorloop krijgt een eigen map.
Voor een nieuw trainingsexperiment begin je weer met een verse runtime.

In [ ]:
import gc
import hashlib
import json
import random
import shutil
import sys
from collections import Counter
from datetime import datetime, timezone
from importlib.metadata import version
from pathlib import Path

import torch
from datasets import Dataset
from huggingface_hub import model_info
from peft import LoraConfig, PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer, GenerationConfig, set_seed
from trl import SFTConfig, SFTTrainer

if not torch.cuda.is_available():
    raise RuntimeError("Selecteer in Colab een GPU-runtime en voer de cellen opnieuw uit.")

SEED = 42
MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"
MODEL_REVISION = model_info(MODEL_ID).sha
USE_BF16 = torch.cuda.is_bf16_supported()
DTYPE = torch.bfloat16 if USE_BF16 else torch.float16
MAX_LENGTH = 768
EPOCHS = 4
LEARNING_RATE = 2e-4
RUN_DIR = Path("/content/remember-tiny") / datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
RUN_DIR.mkdir(parents=True, exist_ok=False)
set_seed(SEED)

print("GPU:", torch.cuda.get_device_name(0))
print("Model:", MODEL_ID, MODEL_REVISION)
print("Resultaten:", RUN_DIR)

## 3. Exact bepalen wat het model moet leren

| Eerdere context | Gesprek tot nu toe | Gewenst |
|---|---|---|
| Project A: deadline 12 maart | “We moeten op 14 maart klaar zijn.” | MELD |
| Project A: deadline 12 maart | “We moeten op 12 maart klaar zijn.” | STIL |
| Project A: deadline 12 maart | “14 maart… sorry, 12 maart.” | STIL |
| Project A: deadline 12 maart | “We besluiten nu de deadline naar 14 maart te verplaatsen.” | STIL |

Dit zijn onze productregels voor deze proef. `MELD` betekent alleen dat er een
correctie nodig lijkt; het model formuleert die correctie nog niet.

De afspraak wordt telkens als **input** meegegeven. We trainen het gedrag van
vergelijken en zwijgen; de projectdeadlines zelf hoeven niet in de modelgewichten.

In [ ]:
SYSTEM_PROMPT = """Je controleert uitspraken over projectdeadlines met de meegegeven context.
Antwoord alleen MELD of STIL.
MELD: een actuele bewering over de deadline van hetzelfde project botst met de laatst bekende afspraak.
STIL: de bewering klopt, is al gecorrigeerd, gaat over een ander project, is alleen een vraag of hypothetisch voorstel, of de context geeft onvoldoende bewijs.
Een expliciet nieuw besluit in het gesprek vervangt de eerdere afspraak: dan STIL.
Beoordeel het gesprek tot nu toe; verzin geen toekomstige uitspraken."""


def messages_for(example):
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"Eerdere context:\n{example['context']}\n\nGesprek tot nu toe:\n{example['conversation']}"},
    ]


def as_training_row(example):
    return {
        "prompt": messages_for(example),
        "completion": [{"role": "assistant", "content": example["expected"]}],
    }

## 4. Synthetische trainingsdata maken

Hier betekent synthetisch: fictieve scenario's die een Python-generator maakt uit
zelf geschreven patronen. We gebruiken **96 trainingsvoorbeelden** en **16
validatievoorbeelden**. Lees eerst een paar voorbeelden en controleer zelf het label.

- **Train:** de voorbeelden waarmee de adapter wordt aangepast.
- **Validatie:** andere projecten waarmee we het gedrag en de loss tijdens ontwikkeling controleren.
- **Test:** de apart geschreven voorbeelden uit de volgende cel, uitsluitend voor de eindvergelijking.

Train en validatie delen hier bewust dezelfde zinsbouw. Dit is een kleine oefening;
een lage validatieloss bewijst daardoor nog geen brede generalisatie.
Later kan een groter taalmodel meer uiteenlopende scenario's genereren. Dat is een
uitbreiding op deze data-stap; de rest van de keten blijft hetzelfde.

In [ ]:
def make_examples(project, agreed, other):
    variants = [
        ("MELD", "bewering", f"De deadline voor {project} is {other}."),
        ("MELD", "planning", f"Ik plan de oplevering van {project} op {other}; dat is onze deadline."),
        ("MELD", "antwoord", f"A: Wanneer is de deadline van {project}? B: {other}, dat hebben we afgesproken."),
        ("MELD", "bericht", f"Ik schrijf aan de klant dat {project} uiterlijk {other} klaar moet zijn, volgens de bestaande afspraak."),
        ("STIL", "klopt", f"De deadline voor {project} is {agreed}."),
        ("STIL", "zelfcorrectie", f"De deadline van {project} is {other}. Sorry, ik bedoel {agreed}."),
        ("STIL", "nieuw-besluit", f"We besluiten nu de deadline van {project} van {agreed} naar {other} te verplaatsen. Akkoord."),
        ("STIL", "ander-project", f"Over een ander project, Extern: daarvoor is de deadline {other}."),
    ]
    return [
        {"id": f"{project}-{index}", "group": project,
         "context": f"Project {project}: de vastgelegde deadline is {agreed}.",
         "conversation": conversation, "expected": expected, "category": category}
        for index, (expected, category, conversation) in enumerate(variants)
    ]


train_projects = [
    ("Amber", "3 februari", "6 februari"), ("Berk", "8 maart", "11 maart"),
    ("Ceder", "12 april", "15 april"), ("Duin", "17 mei", "20 mei"),
    ("Esdoorn", "2 juni", "5 juni"), ("Fakkel", "9 juli", "12 juli"),
    ("Golf", "14 augustus", "17 augustus"), ("Horizon", "4 september", "7 september"),
    ("Iris", "11 oktober", "14 oktober"), ("Jasmijn", "18 november", "21 november"),
    ("Koraal", "5 december", "8 december"), ("Linde", "16 januari", "19 januari"),
]
valid_projects = [("Maan", "7 februari", "10 februari"), ("Noord", "22 maart", "25 maart")]
train_examples = [example for args in train_projects for example in make_examples(*args)]
valid_examples = [example for args in valid_projects for example in make_examples(*args)]
random.Random(SEED).shuffle(train_examples)

for example in make_examples(*train_projects[0]):
    print(example["context"], "|", example["conversation"], "→", example["expected"])

print("Aantallen:", len(train_examples), "train,", len(valid_examples), "validatie")
print(json.dumps(as_training_row(train_examples[0]), ensure_ascii=False, indent=2))

## 5. Een kleine eigen benchmark

Hieronder staan **24 apart geschreven testgevallen**, gegroepeerd in 12 nieuwe
projecten. Binnen elk project staat een paar met verschillende verwachte uitkomsten.
Zo controleren we onder meer of de context daadwerkelijk verschil maakt.

De test bevat andere formuleringen, voorstellen, gewijzigde besluiten en ontbrekend
bewijs. De 24 gevallen zijn geen 24 onafhankelijke echte gesprekken: rapporteer ook
de 12 projectgroepen. De getallen zijn bedoeld om de meetketen te begrijpen.

Leg de data en instellingen vóór de nulmeting vast. Pas tijdens deze eerste run
niets aan op basis van testresultaten. Voor verdere experimenten gebruik je validatie;
na herhaald afstemmen op deze test heb je een nieuwe, ongeziene eindtest nodig.

In [ ]:
test_pairs = [
    ("Olijf", [
        ("Olijf moet op 12 april worden opgeleverd.", "Voor Olijf staat bij mij 14 april als harde einddatum.", "MELD", "context-verschil"),
        ("Olijf moet op 14 april worden opgeleverd.", "Voor Olijf staat bij mij 14 april als harde einddatum.", "STIL", "context-verschil"),
    ]),
    ("Parel", [
        ("Parel: deadline 25 mei.", "Parel moet de twintigste mei af zijn; dat is wat er is vastgelegd.", "MELD", "uitgeschreven-datum"),
        ("Parel: deadline 25 mei.", "Parel moet de vijfentwintigste mei af zijn; dat is wat er is vastgelegd.", "STIL", "uitgeschreven-datum"),
    ]),
    ("Quartz", [
        ("Quartz: opleverdatum 6 juni.", "Quartz is op 6 juni klaar. Nee wacht, 9 juni, dat is de afgesproken deadline.", "MELD", "laatste-bewering"),
        ("Quartz: opleverdatum 6 juni.", "Quartz is op 9 juni klaar. Nee wacht, 6 juni, dat is de afgesproken deadline.", "STIL", "laatste-bewering"),
    ]),
    ("Rivier", [
        ("Rivier: vastgestelde deadline 3 juli.", "Voor Rivier blijft alles zoals afgesproken: 8 juli is de deadline.", "MELD", "besluit-of-bewering"),
        ("Rivier: vastgestelde deadline 3 juli.", "De klant stemt nu in met uitstel van Rivier tot 8 juli. Daarmee vervangen we de oude deadline.", "STIL", "besluit-of-bewering"),
    ]),
    ("Ster", [
        ("Ster: uiterlijk 10 augustus gereed.", "Dan Ster: daarvoor hebben we 15 augustus als uiterste datum staan.", "MELD", "project-scope"),
        ("Ster: uiterlijk 10 augustus gereed.", "Dan een ander project, Prisma: daarvoor hebben we 15 augustus als uiterste datum staan.", "STIL", "project-scope"),
    ]),
    ("Toermalijn", [
        ("Toermalijn: de afgesproken deadline is 2 september.", "Toermalijn heeft volgens onze afspraak 5 september als deadline.", "MELD", "hypothese"),
        ("Toermalijn: de afgesproken deadline is 2 september.", "Stel dat we Toermalijn tot 5 september uitstellen? Het is een gedachte, er is nog niets gewijzigd.", "STIL", "hypothese"),
    ]),
    ("Uil", [
        ("Uil: eerst was 4 oktober afgesproken. Daarna is definitief 9 oktober besloten.", "De huidige deadline voor Uil is dus 4 oktober.", "MELD", "laatste-afspraak"),
        ("Uil: eerst was 4 oktober afgesproken. Daarna is definitief 9 oktober besloten.", "De huidige deadline voor Uil is dus 9 oktober.", "STIL", "laatste-afspraak"),
    ]),
    ("Vonk", [
        ("Vonk: de deadline is 7 november.", "A: Wanneer is Vonk af? B: De vastgelegde datum is 11 november.", "MELD", "vraag-of-antwoord"),
        ("Vonk: de deadline is 7 november.", "A: Welke datum hadden we voor Vonk eigenlijk als deadline?", "STIL", "vraag-of-antwoord"),
    ]),
    ("Wolk", [
        ("Wolk: deadline 1 december.", "Wolk moet op 4 december klaar zijn; dat is de afspraak.", "MELD", "bewijs-aanwezig"),
        ("Wolk: de deadline is niet vastgelegd in deze notities.", "Wolk moet op 4 december klaar zijn; dat is de afspraak.", "STIL", "bewijs-aanwezig"),
    ]),
    ("Zilver", [
        ("Zilver: de huidige deadline is 20 januari, eerder was het 15 januari.", "Die 20 januari komt uit oude notities. De huidige deadline van Zilver is 15 januari.", "MELD", "historische-datum"),
        ("Zilver: de huidige deadline is 20 januari, eerder was het 15 januari.", "Die 15 januari komt uit oude notities. De huidige deadline van Zilver is 20 januari.", "STIL", "historische-datum"),
    ]),
    ("Yuzu", [
        ("Yuzu: deadline 13 februari.", "In ons overzicht staat Yuzu op 18 februari als opleverdeadline; daar houd ik mij aan.", "MELD", "ontbrekende-context"),
        ("Yuzu: Sam is de ontwerper; er staat geen opleverdatum bij.", "In ons overzicht staat Yuzu op 18 februari als opleverdeadline; daar houd ik mij aan.", "STIL", "ontbrekende-context"),
    ]),
    ("Zee", [
        ("Zee: presentatie op 21 maart, deadline voor oplevering op 16 maart.", "De opleverdeadline voor Zee is 21 maart.", "MELD", "datum-betekenis"),
        ("Zee: presentatie op 21 maart, deadline voor oplevering op 16 maart.", "De opleverdeadline voor Zee is 16 maart.", "STIL", "datum-betekenis"),
    ]),
]
test_examples = [
    {"id": f"{project}-{index}", "group": project, "context": context,
     "conversation": conversation, "expected": expected, "category": category}
    for project, pairs in test_pairs
    for index, (context, conversation, expected, category) in enumerate(pairs)
]

splits = {"train": train_examples, "valid": valid_examples, "test": test_examples}
groups = {name: {row["group"] for row in rows} for name, rows in splits.items()}
assert groups["train"].isdisjoint(groups["valid"] | groups["test"])
assert groups["valid"].isdisjoint(groups["test"])
all_examples = train_examples + valid_examples + test_examples
assert len({row["id"] for row in all_examples}) == len(all_examples)
assert len({(row["context"], row["conversation"]) for row in all_examples}) == len(all_examples)
assert all(row["expected"] in {"MELD", "STIL"} for row in all_examples)

DATA_HASHES = {}
for name, rows in splits.items():
    payload = "\n".join(json.dumps(row, ensure_ascii=False, sort_keys=True) for row in rows) + "\n"
    (RUN_DIR / f"{name}.jsonl").write_text(payload, encoding="utf-8")
    DATA_HASHES[name] = hashlib.sha256(payload.encode()).hexdigest()
    print(name, len(rows), "voorbeelden,", len(groups[name]), "projecten,", dict(Counter(row["expected"] for row in rows)))

## 6. Het bestaande model laden

Hugging Face levert hier de gewichten en de **tokenizer**: de omzetter van tekst
naar modeltokens. Het model wordt geladen in het GPU-geheugen van Colab.
Er is nog niets getraind. Je kunt de onderstaande twee voorbeelden zelf veranderen.

We gebruiken dezelfde instructie en dezelfde manier van antwoorden voor de
nulmeting en het getrainde model. Onverwachte tekst telt als ongeldig antwoord;
de evaluator haalt er niet achteraf een gunstig label uit.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=MODEL_REVISION)
tokenizer.padding_side = "right"
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, revision=MODEL_REVISION, torch_dtype=DTYPE, attn_implementation="sdpa",
).to("cuda")

generation_config = GenerationConfig(
    max_new_tokens=8, do_sample=False, use_cache=True,
    eos_token_id=tokenizer.eos_token_id, pad_token_id=tokenizer.pad_token_id,
)


@torch.inference_mode()
def predict(active_model, example):
    active_model.eval()
    inputs = tokenizer.apply_chat_template(
        messages_for(example), tokenize=True, add_generation_prompt=True,
        return_dict=True, return_tensors="pt",
    ).to(active_model.device)
    output = active_model.generate(**inputs, generation_config=generation_config)
    return tokenizer.decode(output[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()


for example in valid_examples[:2]:
    print(example["conversation"], "→", predict(model, example), "(verwacht:", example["expected"], ")")

## 7. De nulmeting uitvoeren

Onze benchmark is gewoon Python die dezelfde vragen stelt en de antwoorden telt.

- **Precision MELD:** welk deel van de meldingen is terecht?
- **Recall MELD:** welk deel van de benodigde meldingen doet het model?
- **Onnodige meldingen:** het aantal keer MELD terwijl STIL nodig was.
- **Ongeldige antwoorden:** bijvoorbeeld een uitleg in plaats van één label.
- **Accuracy:** het aandeel exacte goede antwoorden; bekijk dit samen met de andere maten.

Bij nul meldingen is precision niet gedefinieerd (`None`). Ongeldige antwoorden
blijven fouten, ook als ze toevallig een gewenst woord bevatten. Een model dat altijd
STIL antwoordt haalt hier 50% accuracy en 0% recall; dat is onze eenvoudige controle.

De voorspellingen vóór training worden opgeslagen. Gebruik de test niet om
vervolgens de instellingen van deze run te kiezen.

In [ ]:
def metrics(predictions):
    tp = sum(row["expected"] == "MELD" and row["prediction"] == "MELD" for row in predictions)
    fp = sum(row["expected"] == "STIL" and row["prediction"] == "MELD" for row in predictions)
    positives = sum(row["expected"] == "MELD" for row in predictions)
    negatives = sum(row["expected"] == "STIL" for row in predictions)
    correct = sum(row["expected"] == row["prediction"] for row in predictions)
    return {
        "voorbeelden": len(predictions),
        "correct": correct,
        "accuracy": correct / len(predictions),
        "precision_meld": tp / (tp + fp) if tp + fp else None,
        "recall_meld": tp / positives if positives else None,
        "onnodige_meldingen": fp,
        "aandeel_onnodig_bij_stil": fp / negatives if negatives else None,
        "ongeldige_antwoorden": sum(row["prediction"] not in {"MELD", "STIL"} for row in predictions),
    }


def evaluate(active_model, examples):
    return [{**example, "prediction": predict(active_model, example)} for example in examples]


def save_json(name, value):
    (RUN_DIR / name).write_text(json.dumps(value, ensure_ascii=False, indent=2), encoding="utf-8")

In [ ]:
always_silent = [{**example, "prediction": "STIL"} for example in test_examples]
before = evaluate(model, test_examples)
valid_before = evaluate(model, valid_examples)
save_json("before.json", before)
print("Altijd stil:", metrics(always_silent))
print("Nulmeting test:", metrics(before))
print("Nulmeting validatie:", metrics(valid_before))

## 8. Echt trainen: SFT met LoRA

**SFT** is supervised fine-tuning: we tonen de input en het gewenste antwoord.
De trainer berekent hoeveel het model afwijkt en past trainbare gewichten aan.
`completion_only_loss=True` gebruikt ons `completion_mask`: de prompt, inclusief
de assistant-kop, telt niet mee in de loss. Het antwoord en de afsluitende
chatopmaak tellen wel mee. We controleren de zichtbare antwoordtekst vóór de training.

**LoRA** bepaalt welke gewichten veranderen: kleine toegevoegde matrices, de
**adapter**. De bestaande modelgewichten blijven tijdens deze training bevroren.
SFT is dus de trainingsmethode; LoRA is de manier om weinig parameters aan te passen.

We kiezen vooraf vier rondes door de 96 voorbeelden (`epochs=4`), een learning rate
van `0.0002` en LoRA-rank `8`. De rank bepaalt de omvang van de aanpassing.
`print_trainable_parameters()` laat zien hoeveel parameters werkelijk worden getraind.
De validatieset wordt elke ronde gebruikt om loss te meten; de testset komt niet in de trainer.

Een dalende loss betekent dat de gewenste antwoorden waarschijnlijker worden.
Of het gedrag bruikbaarder wordt, meten we daarna met de benchmark.

In [ ]:
def tokenize_training_row(example):
    row = as_training_row(example)
    # De assistant-kop hoort al bij de input, net als bij predict().
    prompt_ids = tokenizer.apply_chat_template(
        row["prompt"], tokenize=True, add_generation_prompt=True,
    )
    input_ids = tokenizer.apply_chat_template(
        row["prompt"] + row["completion"], tokenize=True, add_generation_prompt=False,
    )
    assert input_ids[:len(prompt_ids)] == prompt_ids, "Prompt en trainingsreeks sluiten niet aan"
    assert len(input_ids) <= MAX_LENGTH, f"Voorbeeld te lang: {len(input_ids)} tokens"
    return {
        "input_ids": input_ids,
        "completion_mask": [0] * len(prompt_ids) + [1] * (len(input_ids) - len(prompt_ids)),
    }


train_dataset = Dataset.from_list([tokenize_training_row(row) for row in train_examples])
valid_dataset = Dataset.from_list([tokenize_training_row(row) for row in valid_examples])

lora_config = LoraConfig(
    task_type="CAUSAL_LM", r=8, lora_alpha=16, lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"], bias="none", revision=MODEL_REVISION,
)
training_config = SFTConfig(
    output_dir=str(RUN_DIR / "training"),
    num_train_epochs=EPOCHS, learning_rate=LEARNING_RATE,
    per_device_train_batch_size=2, per_device_eval_batch_size=2,
    gradient_accumulation_steps=4, gradient_checkpointing=False,
    max_length=MAX_LENGTH, packing=False, completion_only_loss=True,
    eval_strategy="epoch", save_strategy="no", logging_steps=6,
    bf16=USE_BF16, fp16=not USE_BF16, optim="adamw_torch",
    lr_scheduler_type="constant", report_to="none", push_to_hub=False,
    seed=SEED, data_seed=SEED,
)
model.config.use_cache = False
trainer = SFTTrainer(
    model=model, args=training_config, peft_config=lora_config,
    train_dataset=train_dataset, eval_dataset=valid_dataset, processing_class=tokenizer,
)
trainer.model.print_trainable_parameters()
assert all(not param.requires_grad for name, param in trainer.model.named_parameters() if "lora_" not in name)
first_batch = trainer.data_collator([trainer.train_dataset[0]])
target_ids = first_batch["labels"][0]
target_text = tokenizer.decode(target_ids[target_ids != -100], skip_special_tokens=True).strip()
assert target_text in {"MELD", "STIL"}, f"Controleer de loss-maskering: {target_text!r}"
print("De loss wordt berekend op antwoordtokens:", target_text)
del first_batch, target_ids
trainer.train()
trained_model = trainer.model

## 9. Precies dezelfde benchmark herhalen

We veranderen geen prompt, testvragen of beoordelingsregels. Hieronder staan de
werkelijke resultaten van jouw run; er zijn geen vooraf ingevulde scores.
Bekijk de individuele veranderingen: winst op outputformaat is iets anders dan
winst op het herkennen van een tegenstrijdigheid.

**Bij geen verbetering:** dat is een geldige uitkomst. Bekijk fouten op validatie,
formuleer één hypothese (bijvoorbeeld: te weinig voorbeelden van gewijzigde besluiten),
maak een nieuwe run en verander één ding. Ga niet eindeloos op deze kleine test afstemmen.

In [ ]:
after = evaluate(trained_model, test_examples)
valid_after = evaluate(trained_model, valid_examples)
summary = {
    "altijd_stil": metrics(always_silent),
    "test_voor": metrics(before), "test_na": metrics(after),
    "validatie_voor": metrics(valid_before), "validatie_na": metrics(valid_after),
}
save_json("after.json", after)
save_json("validation-before.json", valid_before)
save_json("validation-after.json", valid_after)
save_json("metrics.json", summary)
print(json.dumps(summary, ensure_ascii=False, indent=2))

for old, new in zip(before, after):
    assert old["id"] == new["id"]
    print(f"{new['id']:12} verwacht={new['expected']:4} voor={old['prediction']!r} na={new['prediction']!r}")

## 10. Het resultaat opslaan

Het belangrijkste trainingsresultaat is `adapter/adapter_model.safetensors`, samen
met `adapter_config.json`. Dit zijn de geleerde aanpassingen. Voor gebruik heb je ook
het oorspronkelijke Qwen-model nodig; een adapter is niet zelfstandig het hele model.

We bewaren ook data, prompt, instellingen, libraryversies, modelrevisie en scores.
Daarmee kun je later terugvinden wat je precies hebt gedaan.

In [ ]:
ADAPTER_DIR = RUN_DIR / "adapter"
trained_model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
save_json("training-log.json", trainer.state.log_history)
manifest = {
    "model_id": MODEL_ID, "model_revision": MODEL_REVISION,
    "system_prompt": SYSTEM_PROMPT, "seed": SEED,
    "data_sha256": DATA_HASHES,
    "gpu": torch.cuda.get_device_name(0), "torch": torch.__version__, "python": sys.version,
    "packages": {name: version(name) for name in ["transformers", "trl", "peft", "datasets", "accelerate", "huggingface-hub"]},
    "training": json.loads(training_config.to_json_string()),
    "lora": json.loads((ADAPTER_DIR / "adapter_config.json").read_text(encoding="utf-8")),
    "generation": generation_config.to_dict(),
}
save_json("manifest.json", manifest)
print("Adapterbestanden:", [path.name for path in ADAPTER_DIR.iterdir()])

## 11. De opgeslagen adapter opnieuw gebruiken

We verwijderen de modellen uit het werkgeheugen, laden hetzelfde basismodel opnieuw
en koppelen de opgeslagen adapter eraan. Eén eerder getest voorbeeld controleert of
opslaan en terugladen hetzelfde antwoord geeft. Daarna proberen we een nieuw fictief geval.

Dit is de interface die een app uiteindelijk nodig heeft: context + gesprek erin,
modelbeslissing eruit. Audio, ophalen van context en timing van onderbrekingen zijn
volgende experimenten.

In [ ]:
reload_example = test_examples[0]
expected_reload_output = after[0]["prediction"]
del trainer, trained_model, model
gc.collect()
torch.cuda.empty_cache()

saved = json.loads((RUN_DIR / "manifest.json").read_text(encoding="utf-8"))
SYSTEM_PROMPT = saved["system_prompt"]
tokenizer = AutoTokenizer.from_pretrained(ADAPTER_DIR)
generation_config = GenerationConfig.from_dict(saved["generation"])
fresh_base = AutoModelForCausalLM.from_pretrained(
    saved["model_id"], revision=saved["model_revision"],
    torch_dtype=DTYPE, attn_implementation="sdpa",
).to("cuda")
reloaded_model = PeftModel.from_pretrained(fresh_base, ADAPTER_DIR)
assert predict(reloaded_model, reload_example) == expected_reload_output, "Het antwoord na terugladen verschilt; onderzoek dit voordat je het resultaat gebruikt."

new_example = {
    "context": "Project Haven: de vastgelegde deadline is 12 maart.",
    "conversation": "We hebben voor Haven 14 maart als deadline afgesproken.",
}
print("Opnieuw geladen model:", predict(reloaded_model, new_example))

## 12. Alles meenemen

Een Colab-runtime is tijdelijk. Download de ZIP en bewaar ook het uitgevoerde
notebook via *File → Download → Download .ipynb*. De ZIP bevat de adapter en run-data;
het basismodel wordt later opnieuw van Hugging Face geladen.

Als je klaar bent: *Runtime → Disconnect and delete runtime*.

In [ ]:
from google.colab import files

archive_path = shutil.make_archive(str(RUN_DIR), "zip", root_dir=RUN_DIR)
files.download(archive_path)

## Wat je na deze oefening moet kunnen uitleggen

1. Welk gedrag train je, en welke informatie geef je bij gebruik gewoon als context mee?
2. Welke data hebben de gewichten veranderd, en welke data waren alleen voor evaluatie?
3. Wat doet SFT, en wat doet LoRA?
4. Wat is het verschil tussen een lagere loss en minder onnodige meldingen?
5. Welke bestanden heb je nodig om jouw getrainde model weer te gebruiken?

Een volgende stap is een grotere, gevarieerdere dataset, gemaakt met een taalmodel
en gecontroleerd op labelkwaliteit. Daarna volgen echte transcriptfragmenten en een
sterker klein model. De structuur van dit notebook blijft daarbij bruikbaar.

## Bronnen

- [Google Colab: runtime, upload en beschikbaarheid](https://research.google.com/colaboratory/faq.html)
- [Qwen2.5-0.5B-Instruct modelkaart](https://huggingface.co/Qwen/Qwen2.5-0.5B-Instruct)
- [TRL SFTTrainer 0.23.1](https://huggingface.co/docs/trl/v0.23.1/en/sft_trainer)
- [PEFT: adapters trainen, bewaren en laden](https://huggingface.co/docs/peft/en/quicktour)

Dit notebook is voorbereid met uitleg en vaste begininstellingen. De GPU-training
is pas uitgevoerd wanneer jij de trainingscel draait; lege uitvoervakken zijn geen meetresultaten.